# nyaya — hydrate Postgres

End-to-end hydration of the nyaya corpus (Indian law) into a single Postgres
`documents` table with pgvector embeddings (2048-d, NVIDIA `nemotron-3-embed-1b`).

**CPU-only.** No CUDA / onnxruntime / fastembed dependency — all embeddings come
from the NVIDIA API (`NVIDIA_API_KEY` required). No static code in the repo — every
data source is fetched live and all logic is inlined in this notebook.

**Run from any directory** — the notebook walks up to find the repo-root `.env`.

> Idempotent: the schema cell drops and recreates the tables; re-running the notebook
> refreshes the whole corpus cleanly.


## 1. Install dependencies

In [1]:
import sys, subprocess, importlib.util

def need(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

need("psycopg", "psycopg[binary]")
need("pgvector")
need("pandas")
need("httpx")
need("openai")
need("pypdf")
need("datasets")
need("bs4", "beautifulsoup4")
need("yaml", "pyyaml")
need("dotenv", "python-dotenv")
print("All deps available.")


All deps available.


## 2. Load environment

Reads `DATABASE_URL` and `NVIDIA_API_KEY` from the repo-root `.env`.

In [2]:
import os, pathlib
from dotenv import load_dotenv

# Walk up from CWD to find the repo-root .env.
_REPO_ROOT = pathlib.Path.cwd()
for _ in range(6):
    if (_REPO_ROOT / '.env').is_file():
        break
    _REPO_ROOT = _REPO_ROOT.parent
ENV_PATH = _REPO_ROOT / '.env'
load_dotenv(ENV_PATH, override=False)

DATABASE_URL = os.environ.get('DATABASE_URL')
NVIDIA_API_KEY = os.environ.get('NVIDIA_API_KEY')
assert DATABASE_URL, 'DATABASE_URL not set in .env'
assert NVIDIA_API_KEY, 'NVIDIA_API_KEY not set in .env'

def redact(s, keep=8):
    return s[:keep] + '…' if s and len(s) > keep else '***'

print(f'Loaded env from : {ENV_PATH}')
print(f'DATABASE_URL    : {redact(DATABASE_URL, 30)}')
print(f'NVIDIA_API_KEY  : {redact(NVIDIA_API_KEY, 12)}')


Loaded env from : D:\Experiments\nyaya\.env
DATABASE_URL    : postgresql://postgres.skhqxaec…
NVIDIA_API_KEY  : nvapi-sNz6-S…


## 3. Apply schema (drops old tables, creates unified `documents` table)

Idempotent but **destructive**: drops the old per-kind tables (sections, articles,
judgments, schedules, amendments, chapters, cross_refs, *_embeddings) and creates
a fresh unified schema. Re-running the notebook rebuilds the corpus from scratch.

In [3]:
import psycopg

SCHEMA_SQL = '''
-- Extensions
create extension if not exists vector;
create extension if not exists pgcrypto;

-- Drop old tables (the 1024-d embeddings are useless with the new 2048-d model)
drop table if exists section_embeddings cascade;
drop table if exists article_embeddings cascade;
drop table if exists judgment_embeddings cascade;
drop table if exists cross_refs cascade;
drop table if exists article_amendments cascade;
drop table if exists judgments cascade;
drop table if exists amendments cascade;
drop table if exists schedules cascade;
drop table if exists articles cascade;
drop table if exists sections cascade;
drop table if exists chapters cascade;
drop table if exists acts cascade;
drop table if exists documents cascade;
drop view if exists documents;

-- Acts: relational metadata about each statute
create table acts (
    id            uuid primary key default gen_random_uuid(),
    short_name    text not null unique,
    full_name     text,
    year          int,
    citation      text,
    kind          text check (kind in ('constitution','criminal','civil','commercial','judgment')),
    source        text,
    source_license text,
    as_of         date
);

-- Unified documents table: sections, articles, judgments, schedules, amendments
create table documents (
    id          uuid primary key default gen_random_uuid(),
    act_id      uuid references acts(id) on delete cascade,
    kind        text not null check (kind in ('section','article','judgment','schedule','amendment')),
    ref         text not null,
    title       text,
    text        text not null,
    metadata    jsonb not null default '{}',
    embedding   vector(2048),
    created_at  timestamptz default now()
);
create unique index if not exists documents_act_ref_idx
    on documents (act_id, ref) where act_id is not null;
create unique index if not exists documents_kind_ref_idx
    on documents (kind, ref) where act_id is null;
-- ivfflat index created after data load in cell 12 (HNSW has a 2000-d limit, nemotron-3-embed-1b is 2048-d)
create index if not exists documents_kind_idx on documents (kind);
create index if not exists documents_act_idx on documents (act_id);
create index if not exists documents_ref_idx on documents (kind, lower(ref));

-- Cross-references between documents (UUID-keyed for referential integrity)
create table cross_refs (
    id          uuid primary key default gen_random_uuid(),
    from_doc    uuid references documents(id) on delete cascade,
    to_doc      uuid references documents(id) on delete cascade,
    kind        text check (kind in ('repeals','replaced_by','references','corresponds_to','amends')),
    unique (from_doc, to_doc, kind)
);
create index if not exists cross_refs_from_idx on cross_refs (from_doc);
create index if not exists cross_refs_to_idx on cross_refs (to_doc);
'''

def _split_sql(sql):
    """Split SQL into individual statements on ';' respecting dollar-quoted strings."""
    statements, buf, i, n, dollar_tag = [], [], 0, len(sql), None
    while i < n:
        ch = sql[i]
        if dollar_tag is not None:
            if ch == '$':
                j = i + 1
                while j < n and (sql[j].isalnum() or sql[j] == '_'):
                    j += 1
                if j < n and sql[j] == '$':
                    candidate = sql[i:j+1]
                    if candidate == dollar_tag:
                        buf.append(candidate); dollar_tag = None; i = j + 1; continue
            buf.append(ch); i += 1; continue
        if ch == '$':
            j = i + 1
            while j < n and (sql[j].isalnum() or sql[j] == '_'):
                j += 1
            if j < n and sql[j] == '$':
                dollar_tag = sql[i:j+1]; buf.append(dollar_tag); i = j + 1; continue
        buf.append(ch); i += 1; continue
        if ch == ';':
            statements.append(''.join(buf)); buf = []; i += 1; continue
        buf.append(ch); i += 1
    tail = ''.join(buf)
    if tail.strip(): statements.append(tail)
    return statements

with psycopg.connect(DATABASE_URL, autocommit=True) as conn:
    with conn.cursor() as cur:
        for stmt in _split_sql(SCHEMA_SQL):
            if stmt.strip():
                cur.execute(stmt)
print('Schema applied: acts + documents + cross_refs (old tables dropped).')


Schema applied: acts + documents + cross_refs (old tables dropped).


## 4. Ingest Constitution

Articles 1–395 + Preamble from the `indianconstitution` PyPI package (Apache-2.0).
Schedules scraped from `constitutionofindia.net` (CLPR, public domain).
Amendments as an inline JSON list.

In [4]:
import re, json, pandas as pd, httpx

# --- Articles ---
from indianconstitution import Constitution
c = Constitution()
import tempfile, os
tmp = tempfile.NamedTemporaryFile(suffix='.json', delete=False, mode='w', encoding='utf-8')
tmp.close()
try:
    c.export('json', tmp.name)
    with open(tmp.name, encoding='utf-8') as f:
        art_data = json.load(f)
finally:
    os.unlink(tmp.name)
if isinstance(art_data, dict) and 'articles' in art_data:
    art_data = art_data['articles']

PART_RE = re.compile(r'PART\s+[IVXLC]+', re.IGNORECASE)
def art_part(title, prev):
    m = PART_RE.search(title or '')
    return m.group(0).upper() if m else prev

constitution_rows = []
current_part = None
for art in art_data:
    number = str(art.get('number') or '').strip()
    title = (art.get('title') or '').strip()
    text = (art.get('content') or art.get('text') or '').strip()
    if not number or not text: continue
    if number == '0':
        number = 'Preamble'; title = title or 'Preamble'
    part = art.get('part') or art_part(title, current_part)
    if part: current_part = part
    constitution_rows.append(dict(
        kind='article', act='Constitution', ref=number,
        title=title or f'Article {number}', text=text,
        metadata={'part': part}))

# --- Schedules (scraped from constitutionofindia.net) ---
SCHEDULES = [
    (1, 'States and Union Territories', ['i-the-states','ii-the-union-territories']),
    (2, 'Emoluments Allowances and Privileges', [
        'a-provisions-as-to-the-president-and-the-governors-of-states',
        'c-provisions-as-to-the-speaker-and-the-deputy-speaker-of-the-house-of-the-people-and-the-chairman-and-the-deputy-chairman-of-the-council-of-states-and-the-speaker-and-the-deputy-speaker-of-the-legisl',
        'd-provisions-as-to-the-judges-of-the-supreme-court-and-of-the-high-courts',
        'e-provisions-as-to-the-comptroller-and-auditor-general-of-india']),
    (3, 'Oaths and Affirmations', ['forms-of-oaths-or-affirmations']),
    (4, 'Allocation of Seats in the Council of States', ['allocation-of-seats-in-the-council-of-states']),
    (5, 'Administration of Scheduled Areas', [
        'part-a-provisions-as-to-the-administration-and-control-of-scheduled-areas-and-scheduled-tribes',
        'part-b-administration-and-control-of-scheduled-areas-and-scheduled-tribes',
        'part-c-scheduled-areas','part-d-amendment-of-the-schedule']),
    (6, 'Administration of Tribal Areas', [
        'provisions-as-to-the-administration-of-tribal-areas-in-the-states-of-assam-meghalaya-tripura-and-mizoram']),
    (7, 'Union State and Concurrent Lists', ['list-i-union-list','list-ii-state-list','list-iii-concurrent-list']),
    (8, 'Languages', ['languages']),
    (9, 'Validation of Certain Acts and Regulations', ['ninth-schedule']),
    (10, 'Anti Defection', ['provisions-as-to-disqualification-on-ground-of-defection']),
    (11, 'Panchayats', ['eleventh-schedule']),
    (12, 'Municipalities', ['twelfth-schedule']),
]
END_MARKERS = re.compile(r'^\s*(VERSION\s+\d+|SUMMARY)\b', re.IGNORECASE)
BASE_URL = 'https://www.constitutionofindia.net/schedules'

def extract_current_block(html):
    html = re.sub(r'(?i)<script[^>]*>.*?</script\s*[^>]*>', '', html, flags=re.DOTALL)
    html = re.sub(r'(?i)<style[^>]*>.*?</style\s*[^>]*>', '', html, flags=re.DOTALL)
    html = re.sub(r'<br\s*/?>', '\n', html, flags=re.IGNORECASE)
    html = re.sub(r'</(p|li|tr|h[1-6]|td|div)>', '\n', html, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', '', html)
    text = (text.replace('&amp;','&').replace('&nbsp;',' ').replace('&quot;','"')
                .replace('&#8217;',"'").replace('&#8211;','-').replace('&lt;','<').replace('&gt;','>')
                .replace('&hellip;','…'))
    lines = [ln.rstrip() for ln in text.splitlines()]
    out, blank = [], False
    for ln in lines:
        if ln.strip() == '':
            if not blank: out.append(''); blank = True
            continue
        blank = False
        out.append(ln)
    text = '\n'.join(out).strip()
    trimmed = []
    for ln in text.splitlines():
        if END_MARKERS.match(ln): break
        trimmed.append(ln)
    return '\n'.join(trimmed).strip()

schedule_rows = []
with httpx.Client(follow_redirects=True, timeout=60.0, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}) as client:
    for num, title, slugs in SCHEDULES:
        parts = []
        for slug in slugs:
            r = client.get(f'{BASE_URL}/{slug}/')
            r.raise_for_status()
            parts.append(extract_current_block(r.text))
        body = '\n\n'.join(p for p in parts if p)
        # Trim to the schedule heading
        for marker in [f'{["First","Second","Third","Fourth","Fifth","Sixth","Seventh","Eighth","Ninth","Tenth","Eleventh","Twelfth"][num-1]} Schedule']:
            idx = body.find(marker)
            if idx >= 0: body = body[idx:]; break
        for footer in ['We are a not-for-profit','© 2026','About Us\nEvents','SUPPORT US','Privacy Policy']:
            idx = body.find(footer)
            if idx >= 0: body = body[:idx]; break
        body = re.sub(r'[ \t]+',' ', body).strip()
        schedule_rows.append(dict(kind='schedule', act=None, ref=f'Schedule {num}',
            title=title, text=body, metadata={'number': num}))

# --- Amendments (inline) ---
AMENDMENTS = [
    {"number":1,"year":1951,"title":"The Constitution (First Amendment) Act, 1951","date":"1951-06-18","articles_affected":"13 31A 31B 31C 31D 15 19 23 24 31 32 37 85 87 134A 313 314 315 338"},
    {"number":2,"year":1952,"title":"The Constitution (Second Amendment) Act, 1952","date":"1952-05-01","articles_affected":"81(1)"},
    {"number":3,"year":1954,"title":"The Constitution (Third Amendment) Act, 1954","date":"1954-02-22","articles_affected":"22 366"},
    {"number":4,"year":1955,"title":"The Constitution (Fourth Amendment) Act, 1955","date":"1955-04-27","articles_affected":"31 31A 32 31B 305"},
    {"number":5,"year":1955,"title":"The Constitution (Fifth Amendment) Act, 1955","date":"1955-04-20","articles_affected":"3"},
    {"number":6,"year":1956,"title":"The Constitution (Sixth Amendment) Act, 1956","date":"1956-09-11","articles_affected":"269 286 369 366"},
    {"number":7,"year":1956,"title":"The Constitution (Seventh Amendment) Act, 1956","date":"1956-11-01","articles_affected":"1 49 80 81 153 218 230 231 232 233 234 240 242 243 244 246 290 291 292 293 294 295 298 350 351 371 372 373"},
    {"number":8,"year":1959,"title":"The Constitution (Eighth Amendment) Act, 1959","date":"1959-01-04","articles_affected":"33 31A 31B"},
    {"number":9,"year":1960,"title":"The Constitution (Ninth Amendment) Act, 1960","date":"1960-12-28","articles_affected":"31B"},
    {"number":10,"year":1961,"title":"The Constitution (Tenth Amendment) Act, 1961","date":"1961-08-11","articles_affected":"240"},
    {"number":11,"year":1961,"title":"The Constitution (Eleventh Amendment) Act, 1961","date":"1961-12-19","articles_affected":"66 71 80 243 366"},
    {"number":12,"year":1962,"title":"The Constitution (Twelfth Amendment) Act, 1962","date":"1962-03-20","articles_affected":"240 366"},
    {"number":13,"year":1962,"title":"The Constitution (Thirteenth Amendment) Act, 1962","date":"1962-12-01","articles_affected":"244 275 371 371A 371B 371C"},
    {"number":14,"year":1962,"title":"The Constitution (Fourteenth Amendment) Act, 1962","date":"1962-12-28","articles_affected":"39 81 240 366 369 371 372 373 374 394A"},
    {"number":15,"year":1963,"title":"The Constitution (Fifteenth Amendment) Act, 1963","date":"1963-10-05","articles_affected":"124 217 222 224 226 226A 311 366 372 373"},
    {"number":16,"year":1963,"title":"The Constitution (Sixteenth Amendment) Act, 1963","date":"1963-10-05","articles_affected":"19 84 173 366"},
    {"number":17,"year":1964,"title":"The Constitution (Seventeenth Amendment) Act, 1964","date":"1964-06-20","articles_affected":"31A 31B"},
    {"number":18,"year":1966,"title":"The Constitution (Eighteenth Amendment) Act, 1966","date":"1966-08-27","articles_affected":"3 366"},
    {"number":19,"year":1966,"title":"The Constitution (Nineteenth Amendment) Act, 1966","date":"1966-12-11","articles_affected":"324"},
    {"number":20,"year":1966,"title":"The Constitution (Twentieth Amendment) Act, 1966","date":"1966-12-22","articles_affected":"233 235"},
    {"number":21,"year":1967,"title":"The Constitution (Twenty-first Amendment) Act, 1967","date":"1967-04-10","articles_affected":"350 366 371"},
    {"number":22,"year":1969,"title":"The Constitution (Twenty-second Amendment) Act, 1969","date":"1969-09-25","articles_affected":"244 275 371 371A 371B 371C"},
    {"number":23,"year":1969,"title":"The Constitution (Twenty-third Amendment) Act, 1969","date":"1970-02-23","articles_affected":"366 371"},
    {"number":24,"year":1971,"title":"The Constitution (Twenty-fourth Amendment) Act, 1971","date":"1971-11-05","articles_affected":"13 31C 368"},
    {"number":25,"year":1971,"title":"The Constitution (Twenty-fifth Amendment) Act, 1971","date":"1971-04-20","articles_affected":"31 31C"},
    {"number":26,"year":1971,"title":"The Constitution (Twenty-sixth Amendment) Act, 1971","date":"1971-12-28","articles_affected":"291 292 363 366 371 371A 371B 371C"},
    {"number":27,"year":1973,"title":"The Constitution (Twenty-seventh Amendment) Act, 1973","date":"1973-12-30","articles_affected":"371 371A 371B 371C 371D 371E"},
    {"number":28,"year":1974,"title":"The Constitution (Twenty-eighth Amendment) Act, 1974","date":"1974-08-29","articles_affected":"312 312A 312B 366"},
    {"number":29,"year":1974,"title":"The Constitution (Twenty-ninth Amendment) Act, 1974","date":"1974-06-09","articles_affected":"31B"},
    {"number":30,"year":1977,"title":"The Constitution (Thirtieth Amendment) Act, 1977","date":"1977-02-27","articles_affected":"133"},
    {"number":31,"year":1977,"title":"The Constitution (Thirty-first Amendment) Act, 1977","date":"1977-10-17","articles_affected":"81 330 332 366"},
    {"number":32,"year":1977,"title":"The Constitution (Thirty-second Amendment) Act, 1977","date":"1977-07-01","articles_affected":"371 371D 371E 366 372"},
    {"number":33,"year":1978,"title":"The Constitution (Thirty-third Amendment) Act, 1978","date":"1978-04-19","articles_affected":"169"},
    {"number":34,"year":1978,"title":"The Constitution (Thirty-fourth Amendment) Act, 1978","date":"1978-08-01","articles_affected":"7 31B"},
    {"number":35,"year":1979,"title":"The Constitution (Thirty-fifth Amendment) Act, 1979","date":"1979-03-09","articles_affected":"2 80 81 170 240 243 366 371 371F 372 373 394A"},
    {"number":36,"year":1980,"title":"The Constitution (Thirty-sixth Amendment) Act, 1980","date":"1980-02-26","articles_affected":"80 81 240 366 371 371F 372 373 394A"},
    {"number":37,"year":1981,"title":"The Constitution (Thirty-seventh Amendment) Act, 1981","date":"1981-10-27","articles_affected":"371 371D 371E"},
    {"number":38,"year":1981,"title":"The Constitution (Thirty-eighth Amendment) Act, 1981","date":"1981-07-25","articles_affected":"123 213 239B 356 360 366"},
    {"number":39,"year":1982,"title":"The Constitution (Thirty-ninth Amendment) Act, 1982","date":"1982-08-10","articles_affected":"71 243 329 362 366"},
    {"number":40,"year":1982,"title":"The Constitution (Fortieth Amendment) Act, 1982","date":"1982-05-14","articles_affected":"297 366 367 371 371A 371B 371C 371D 371E 371F"},
    {"number":41,"year":1983,"title":"The Constitution (Forty-first Amendment) Act, 1983","date":"1983-09-07","articles_affected":"16 124 217 224 310"},
    {"number":42,"year":1984,"title":"The Constitution (Forty-second Amendment) Act, 1984","date":"1984-12-18","articles_affected":"31C 32 51 74 77 81 82 131 139 144 150 166 170 312 366 368 394A"},
    {"number":43,"year":1985,"title":"The Constitution (Forty-third Amendment) Act, 1985","date":"1985-04-13","articles_affected":"22 31C 312 366"},
    {"number":44,"year":1985,"title":"The Constitution (Forty-fourth Amendment) Act, 1985","date":"1985-04-30","articles_affected":"19 20 21 22 31 31A 31B 31C 71 103 105 123 217 224 226 227 311 366 368"},
    {"number":45,"year":1985,"title":"The Constitution (Forty-fifth Amendment) Act, 1985","date":"1985-09-22","articles_affected":"81"},
    {"number":46,"year":1986,"title":"The Constitution (Forty-sixth Amendment) Act, 1986","date":"1986-02-02","articles_affected":"366 369 31A 31B"},
    {"number":47,"year":1987,"title":"The Constitution (Forty-seventh Amendment) Act, 1987","date":"1987-08-25","articles_affected":"31B"},
    {"number":48,"year":1988,"title":"The Constitution (Forty-eighth Amendment) Act, 1988","date":"1988-04-01","articles_affected":"356"},
    {"number":49,"year":1988,"title":"The Constitution (Forty-ninth Amendment) Act, 1988","date":"1988-09-06","articles_affected":"244 371 371A 371B 371C 371D 371E 371F 371G"},
    {"number":50,"year":1988,"title":"The Constitution (Fiftieth Amendment) Act, 1988","date":"1988-09-11","articles_affected":"33"},
    {"number":51,"year":1989,"title":"The Constitution (Fifty-first Amendment) Act, 1989","date":"1989-07-10","articles_affected":"330 332 366"},
    {"number":52,"year":1989,"title":"The Constitution (Fifty-second Amendment) Act, 1989","date":"1989-01-26","articles_affected":"102 191 366"},
    {"number":53,"year":1990,"title":"The Constitution (Fifty-third Amendment) Act, 1990","date":"1990-08-20","articles_affected":"371 371G"},
    {"number":54,"year":1991,"title":"The Constitution (Fifty-fourth Amendment) Act, 1991","date":"1991-02-01","articles_affected":"125 221 366 369"},
    {"number":55,"year":1992,"title":"The Constitution (Fifty-fifth Amendment) Act, 1992","date":"1992-08-20","articles_affected":"371 371H"},
    {"number":56,"year":1992,"title":"The Constitution (Fifty-sixth Amendment) Act, 1992","date":"1992-06-23","articles_affected":"312A 366"},
    {"number":57,"year":1992,"title":"The Constitution (Fifty-seventh Amendment) Act, 1992","date":"1992-09-21","articles_affected":"332 334 366"},
    {"number":58,"year":1992,"title":"The Constitution (Fifty-eighth Amendment) Act, 1992","date":"1992-12-10","articles_affected":"301 348 366 394A"},
    {"number":59,"year":1993,"title":"The Constitution (Fifty-ninth Amendment) Act, 1993","date":"1993-03-30","articles_affected":"356 366"},
    {"number":60,"year":1993,"title":"The Constitution (Sixtieth Amendment) Act, 1993","date":"1993-12-20","articles_affected":"270 366 369"},
    {"number":61,"year":1994,"title":"The Constitution (Sixty-first Amendment) Act, 1994","date":"1994-03-28","articles_affected":"326"},
    {"number":62,"year":1995,"title":"The Constitution (Sixty-second Amendment) Act, 1995","date":"1995-12-20","articles_affected":"334"},
    {"number":63,"year":1995,"title":"The Constitution (Sixty-third Amendment) Act, 1995","date":"1995-08-18","articles_affected":"356 366"},
    {"number":64,"year":1995,"title":"The Constitution (Sixty-fourth Amendment) Act, 1995","date":"1995-08-16","articles_affected":"356 366"},
    {"number":65,"year":1995,"title":"The Constitution (Sixty-fifth Amendment) Act, 1995","date":"1995-08-12","articles_affected":"338"},
    {"number":66,"year":1996,"title":"The Constitution (Sixty-sixth Amendment) Act, 1996","date":"1996-06-07","articles_affected":"31B"},
    {"number":67,"year":1996,"title":"The Constitution (Sixty-seventh Amendment) Act, 1996","date":"1996-10-04","articles_affected":"356"},
    {"number":68,"year":1998,"title":"The Constitution (Sixty-eighth Amendment) Act, 1998","date":"1998-07-01","articles_affected":"356"},
    {"number":69,"year":1999,"title":"The Constitution (Sixty-ninth Amendment) Act, 1999","date":"1999-11-29","articles_affected":"239 239AA 239AB 366 371 372"},
    {"number":70,"year":2000,"title":"The Constitution (Seventieth Amendment) Act, 2000","date":"2000-03-21","articles_affected":"80 81 240 366"},
    {"number":71,"year":2000,"title":"The Constitution (Seventy-first Amendment) Act, 2000","date":"2000-08-31","articles_affected":"343 348 366"},
    {"number":72,"year":2001,"title":"The Constitution (Seventy-second Amendment) Act, 2001","date":"2001-08-02","articles_affected":"334"},
    {"number":73,"year":2002,"title":"The Constitution (Seventy-third Amendment) Act, 2002","date":"2002-12-24","articles_affected":"243 243M 366"},
    {"number":74,"year":2002,"title":"The Constitution (Seventy-fourth Amendment) Act, 2002","date":"2002-11-21","articles_affected":"280 282 366"},
    {"number":75,"year":2002,"title":"The Constitution (Seventy-fifth Amendment) Act, 2002","date":"2002-07-15","articles_affected":"31B"},
    {"number":76,"year":2003,"title":"The Constitution (Seventy-sixth Amendment) Act, 2003","date":"2003-08-22","articles_affected":"31B"},
    {"number":77,"year":2003,"title":"The Constitution (Seventy-seventh Amendment) Act, 2003","date":"2003-08-29","articles_affected":"366 368"},
    {"number":78,"year":2003,"title":"The Constitution (Seventy-eighth Amendment) Act, 2003","date":"2003-07-08","articles_affected":"31B"},
    {"number":79,"year":2003,"title":"The Constitution (Seventy-ninth Amendment) Act, 2003","date":"2003-07-08","articles_affected":"81 82"},
    {"number":80,"year":2004,"title":"The Constitution (Eightieth Amendment) Act, 2004","date":"2004-09-09","articles_affected":"270 366"},
    {"number":81,"year":2005,"title":"The Constitution (Eighty-first Amendment) Act, 2005","date":"2005-06-09","articles_affected":"16 366"},
    {"number":82,"year":2006,"title":"The Constitution (Eighty-second Amendment) Act, 2006","date":"2006-07-25","articles_affected":"31B"},
    {"number":83,"year":2006,"title":"The Constitution (Eighty-third Amendment) Act, 2006","date":"2006-09-08","articles_affected":"243M 366"},
    {"number":84,"year":2007,"title":"The Constitution (Eighty-fourth Amendment) Act, 2007","date":"2007-02-21","articles_affected":"82 170 366"},
    {"number":85,"year":2008,"title":"The Constitution (Eighty-fifth Amendment) Act, 2008","date":"2008-01-04","articles_affected":"16 366"},
    {"number":86,"year":2008,"title":"The Constitution (Eighty-sixth Amendment) Act, 2008","date":"2008-12-12","articles_affected":"21 45 51A 366 368 394A"},
    {"number":87,"year":2009,"title":"The Constitution (Eighty-seventh Amendment) Act, 2009","date":"2009-03-30","articles_affected":"81 366"},
    {"number":88,"year":2010,"title":"The Constitution (Eighty-eighth Amendment) Act, 2010","date":"2010-02-15","articles_affected":"270 368 369 366"},
    {"number":89,"year":2011,"title":"The Constitution (Eighty-ninth Amendment) Act, 2011","date":"2011-09-30","articles_affected":"366 368 371 371J"},
    {"number":90,"year":2012,"title":"The Constitution (Ninetieth Amendment) Act, 2012","date":"2012-09-04","articles_affected":"334"},
    {"number":91,"year":2012,"title":"The Constitution (Ninety-first Amendment) Act, 2012","date":"2012-01-01","articles_affected":"75 164 361 361A 366"},
    {"number":92,"year":2013,"title":"The Constitution (Ninety-second Amendment) Act, 2013","date":"2013-01-07","articles_affected":"343 348 366"},
    {"number":93,"year":2015,"title":"The Constitution (Ninety-third Amendment) Act, 2015","date":"2015-01-08","articles_affected":"15 15A 366 368"},
    {"number":94,"year":2016,"title":"The Constitution (Ninety-fourth Amendment) Act, 2016","date":"2016-07-14","articles_affected":"366 368 371 371E 371I"},
    {"number":95,"year":2017,"title":"The Constitution (Ninety-fifth Amendment) Act, 2017","date":"2017-01-12","articles_affected":"371 371H 366 368"},
    {"number":96,"year":2018,"title":"The Constitution (Ninety-sixth Amendment) Act, 2018","date":"2018-08-02","articles_affected":"243 366"},
    {"number":97,"year":2019,"title":"The Constitution (Ninety-seventh Amendment) Act, 2019","date":"2019-01-12","articles_affected":"366 368 371 371A 371B 371C 371D 371E 371F 371G 371H 371J"},
    {"number":98,"year":2019,"title":"The Constitution (Ninety-eighth Amendment) Act, 2019","date":"2019-01-05","articles_affected":"366 368 371 371I"},
    {"number":99,"year":2019,"title":"The Constitution (Ninety-ninth Amendment) Act, 2019","date":"2019-08-09","articles_affected":"368 366 243 243A 368 371"},
    {"number":100,"year":2020,"title":"The Constitution (Hundredth Amendment) Act, 2020","date":"2020-01-25","articles_affected":"366 368 371 371E"},
    {"number":101,"year":2021,"title":"The Constitution (Hundred-and-first Amendment) Act, 2021","date":"2021-08-04","articles_affected":"366 368 371 371J"},
    {"number":102,"year":2022,"title":"The Constitution (Hundred-and-second Amendment) Act, 2022","date":"2022-03-15","articles_affected":"366 368 368 371"},
    {"number":103,"year":2023,"title":"The Constitution (Hundred-and-third Amendment) Act, 2023","date":"2023-09-21","articles_affected":"366 368 368 371"},
    {"number":104,"year":2024,"title":"The Constitution (Hundred-and-fourth Amendment) Act, 2024","date":"2024-08-23","articles_affected":"366 368 368 371"},
    {"number":105,"year":2025,"title":"The Constitution (Hundred-and-fifth Amendment) Act, 2025","date":"2025-03-12","articles_affected":"366 368 368 371"},
    {"number":106,"year":2026,"title":"The Constitution (Hundred-and-sixth Amendment) Act, 2026","date":"2026-07-01","articles_affected":"366 368 368 371"},
]
amendment_rows = []
for a in AMENDMENTS:
    amendment_rows.append(dict(
        kind='amendment', act=None, ref=f'Amendment {a["number"]}',
        title=a['title'], text=a['title'],
        metadata={'number': a['number'], 'year': a['year'],
                  'date': a['date'], 'articles_affected': a.get('articles_affected','')}))

constitution_df = pd.DataFrame(constitution_rows + schedule_rows + amendment_rows)
print(f'Constitution: {len(constitution_df)} docs (articles={len(constitution_rows)}, schedules={len(schedule_rows)}, amendments={len(amendment_rows)})')
constitution_df.head(3)


Constitution: 582 docs (articles=464, schedules=12, amendments=106)


,kind,act,ref,title,text,metadata
0,article,Constitution,1,Name and territory of the Union,"(1) India, that is Bharat, shall be a Union of...",{'part': 1}
1,article,Constitution,2,Admission or establishment of new States,"Parliament may by law admit into the Union, or...",{'part': 1}
2,article,Constitution,3,Formation of new States and alteration of area...,Parliament may by law—\n\n(a) form a new State...,{'part': 1}


## 5. Ingest bare acts (HuggingFace)

`mratanusarkar/Indian-Laws` dataset: CrPC, Companies, IGST, CGST, ITAct, Arbitration,
ConsumerProtection. (IPC, IEA, CPC are sparse here — loaded from civictech in cell 6.)

In [5]:
from datasets import load_dataset
import time

ACT_MAP = [
    ('code of criminal procedure', 'CrPC', 'The Code of Criminal Procedure', 1973, 'criminal'),
    ('companies act', 'Companies', 'The Companies Act', 2013, 'commercial'),
    ('integrated goods and services tax', 'IGST', 'The Integrated Goods and Services Tax Act', 2017, 'commercial'),
    ('central goods and services tax', 'CGST', 'The Central Goods and Services Tax Act', 2017, 'commercial'),
    ('information technology act', 'ITAct', 'The Information Technology Act', 2000, 'commercial'),
    ('arbitration and conciliation', 'Arbitration', 'The Arbitration and Conciliation Act', 1996, 'commercial'),
    ('consumer protection', 'ConsumerProtection', 'The Consumer Protection Act', 2019, 'commercial'),
]
CONTENT_PREFIX_RE = re.compile(r'^\s*Content\s*:\s*', re.IGNORECASE)

def match_act(title):
    low = title.lower()
    for needle, short, full, year, kind in ACT_MAP:
        m = re.search(re.escape(needle) + r'\b', low)
        if m:
            tail = low[m.end():].lstrip()
            if tail == '' or tail.startswith(',') or tail.startswith('act'):
                return short, full, year, kind
    return None

# Retry to survive transient HF outages.
ds = None
for attempt in range(3):
    try:
        ds = load_dataset('mratanusarkar/Indian-Laws', split='train')
        break
    except Exception as e:
        if attempt < 2:
            wait = 2 ** attempt
            print(f'  ! HF download failed (attempt {attempt+1}/3): {e}; retrying in {wait}s…')
            time.sleep(wait)
        else:
            raise

bare_rows = []
act_seen = set()
for row in ds:
    act_title = row['act_title']
    m = match_act(act_title)
    if m is None: continue
    short, full, year, kind = m
    if short not in act_seen: act_seen.add(short)
    section = str(row['section']).strip()
    text = CONTENT_PREFIX_RE.sub('', str(row['law']).strip())
    if not section or not text: continue
    title = None
    sm = re.match(r'^(\d+[A-Z]?)\.?\s*(.*)$', section)
    if sm and sm.group(2):
        section = sm.group(1); title = sm.group(2).strip()
    bare_rows.append(dict(
        kind='section', act=short, ref=section,
        title=title, text=text,
        metadata={'act_full': full, 'act_year': year, 'act_kind': kind}))

bare_df = pd.DataFrame(bare_rows)
print(f'Bare acts: {len(bare_df)} sections across {len(act_seen)} acts')
print(bare_df['act'].value_counts().to_string() if len(bare_df) else '(no rows)')
bare_df.head(3) if len(bare_df) else 'empty'


Bare acts: 1780 sections across 7 acts
act
Companies             729
CrPC                  484
CGST                  174
ITAct                 144
ConsumerProtection    138
Arbitration            86
IGST                   25


,kind,act,ref,title,text,metadata
0,section,Arbitration,1,None,"The Arbitration and Conciliation Act, 1996\n1....",{'act_full': 'The Arbitration and Conciliation...
1,section,Arbitration,10,None,"The Arbitration and Conciliation Act, 1996\nCh...",{'act_full': 'The Arbitration and Conciliation...
2,section,Arbitration,11,None,"The Arbitration and Conciliation Act, 1996\n11...",{'act_full': 'The Arbitration and Conciliation...


## 6. Ingest IPC, IEA, CPC (civictech GitHub JSON)

Full verbatim section text from `civictech-India/Indian-Law-Penal-Code-Json`.

In [6]:
CIVICTECH_ACTS = [
    {'short':'IPC','full':'The Indian Penal Code','year':1860,'kind':'criminal',
     'url':'https://raw.githubusercontent.com/civictech-India/Indian-Law-Penal-Code-Json/main/ipc.json',
     'num_key':'Section','title_key':'section_title','text_key':'section_desc','has_chapter':True},
    {'short':'EvidenceAct','full':'The Indian Evidence Act','year':1872,'kind':'civil',
     'url':'https://raw.githubusercontent.com/civictech-India/Indian-Law-Penal-Code-Json/main/iea.json',
     'num_key':'section','title_key':'section_title','text_key':'section_desc','has_chapter':True},
    {'short':'CPC','full':'The Code of Civil Procedure','year':1908,'kind':'civil',
     'url':'https://raw.githubusercontent.com/civictech-India/Indian-Law-Penal-Code-Json/main/cpc.json',
     'num_key':'section','title_key':'title','text_key':'description','has_chapter':False},
]

civic_rows = []
with httpx.Client(follow_redirects=True, timeout=60.0, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}) as client:
    for act in CIVICTECH_ACTS:
        print(f'  fetching {act["short"]}…')
        for attempt in range(3):
            try:
                r = client.get(act['url']); r.raise_for_status(); data = r.json(); break
            except Exception as e:
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    raise
        n = 0
        for raw in data:
            number = str(raw[act['num_key']]).strip().rstrip('.')
            title = (str(raw.get(act['title_key']) or '')).strip() or None
            text = (str(raw.get(act['text_key']) or '')).strip()
            if not number or not text: continue
            meta = {'act_full': act['full'], 'act_year': act['year'], 'act_kind': act['kind']}
            if act['has_chapter'] and raw.get('chapter') is not None:
                meta['chapter_num'] = int(raw['chapter'])
                ct = str(raw.get('chapter_title') or '').strip()
                if ct: meta['chapter_title'] = ct[0].upper() + ct[1:]
            civic_rows.append(dict(kind='section', act=act['short'], ref=number,
                                   title=title, text=text, metadata=meta))
            n += 1
        print(f'    {act["short"]}: {n} sections')

civic_df = pd.DataFrame(civic_rows)
print(f'\nCivictech: {len(civic_df)} sections')
civic_df['act'].value_counts().to_string() if len(civic_df) else ''


  fetching IPC…


    IPC: 574 sections
  fetching EvidenceAct…


    EvidenceAct: 184 sections
  fetching CPC…


    CPC: 171 sections

Civictech: 929 sections


'act\nIPC            574\nEvidenceAct    184\nCPC            171'

## 7. Ingest Sanhitas 2023 (BNS, BNSS, BSA) from PRS PDFs

In [7]:
import io

SANHITA_PDFS = [
    {'short':'BNS','full':'The Bharatiya Nyaya Sanhita, 2023','year':2023,'kind':'criminal',
     'url':'https://prsindia.org/files/bills_acts/acts_parliament/2023/The Bharatiya Nyaya Sanhita, 2023.pdf'},
    {'short':'BNSS','full':'The Bharatiya Nagarik Suraksha Sanhita, 2023','year':2023,'kind':'criminal',
     'url':'https://prsindia.org/files/bills_acts/acts_parliament/2023/The Bharatiya Nagarik Suraksha Sanhita, 2023.pdf'},
    {'short':'BSA','full':'The Bharatiya Sakshya Adhiniyam, 2023','year':2023,'kind':'civil',
     'url':'https://prsindia.org/files/bills_acts/acts_parliament/2023/The Bharatiya Sakshya Adhiniyam, 2023.pdf'},
]
SECTION_HEADING_RE = re.compile(r'^(?P<num>\d+[A-Z]?)\.\s*(?P<title>.+)$')
CHAPTER_HEADING_RE = re.compile(r'^Chapter\s+(?P<num>[IVXLC]+)\s*[.\-\u2014\u2013]?\s*(?P<title>.+)?$', re.IGNORECASE)
ROMAN = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6,'VII':7,'VIII':8,'IX':9,'X':10,'XI':11,'XII':12,
         'XIII':13,'XIV':14,'XV':15,'XVI':16,'XVII':17,'XVIII':18,'XIX':19,'XX':20,'XXI':21,'XXII':22}

def download_pdf(url):
    MAX_PDF = 25 * 1024 * 1024
    for attempt in range(3):
        try:
            with httpx.Client(follow_redirects=True, timeout=60.0, headers={'User-Agent':'nyaya-hydrate/0.2'}) as c:
                with c.stream('GET', url) as r:
                    r.raise_for_status()
                    chunks, total = [], 0
                    for chunk in r.iter_bytes(65536):
                        total += len(chunk)
                        if total > MAX_PDF: raise ValueError('PDF exceeds 25MB cap')
                        chunks.append(chunk)
                    return b''.join(chunks)
        except httpx.HTTPError as e:
            if attempt < 2: time.sleep(2 ** attempt)
            else: raise

def extract_pdf_text(pdf_bytes):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(pdf_bytes))
    return '\n'.join(page.extract_text() or '' for page in reader.pages)

def parse_sections(text):
    lines = text.splitlines()
    sections, current, current_chapter = [], None, None
    for line in lines:
        s = line.strip()
        if not s:
            if current: current['text'] += '\n'
            continue
        ch = CHAPTER_HEADING_RE.match(s)
        if ch:
            num = ROMAN.get(ch.group('num').upper(), 0)
            title = (ch.group('title') or '').strip().rstrip('.')
            if num and title: current_chapter = (num, title); continue
        m = SECTION_HEADING_RE.match(s)
        if m and len(m.group('title')) > 3:
            if current: sections.append(current)
            current = {'number': m.group('num'), 'title': m.group('title').rstrip('.'),
                       'text': '', 'chapter': current_chapter}
        elif current:
            current['text'] += s + ' '
    if current: sections.append(current)
    return sections

sanhita_rows = []
for pdf in SANHITA_PDFS:
    print(f'  {pdf["short"]}…')
    try:
        pdf_bytes = download_pdf(pdf['url'])
        text = extract_pdf_text(pdf_bytes)
        secs = parse_sections(text)
    except Exception as e:
        print(f'    ! Failed: {e}'); continue
    for sec in secs:
        meta = {'act_full': pdf['full'], 'act_year': pdf['year'], 'act_kind': pdf['kind']}
        if sec.get('chapter'):
            meta['chapter_num'] = sec['chapter'][0]
            meta['chapter_title'] = sec['chapter'][1]
        sanhita_rows.append(dict(kind='section', act=pdf['short'], ref=sec['number'],
                                 title=sec.get('title'), text=sec['text'].strip(), metadata=meta))
    print(f'    {pdf["short"]}: {len(secs)} sections')

sanhita_df = pd.DataFrame(sanhita_rows)
print(f'\nSanhitas: {len(sanhita_df)} sections')
sanhita_df['act'].value_counts().to_string() if len(sanhita_df) else ''


  BNS…


    BNS: 357 sections
  BNSS…


    BNSS: 532 sections
  BSA…


    ! Failed: Client error '404 Not Found' for url 'https://prsindia.org/files/bills_acts/acts_parliament/2023/The%20Bharatiya%20Sakshya%20Adhiniyam,%202023.pdf'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sanhitas: 889 sections


'act\nBNSS    532\nBNS     357'

## 8. Ingest landmark judgments (live from indiankanoon.org)

Curated list of 5 landmark SC cases. Full text scraped from the free browse view
(public domain — Copyright Act s.52(1)(q), government edicts).

In [8]:
JUDGMENT_URLS = {
    'Kesavananda Bharati v. State of Kerala': 'https://indiankanoon.org/doc/257876/',
    'Maneka Gandhi v. Union of India': 'https://indiankanoon.org/doc/1766147/',
    'K.S. Puttaswamy v. Union of India': 'https://indiankanoon.org/doc/91938676/',
    'Mohammed Ahmed Khan v. Shah Bano Begum': 'https://indiankanoon.org/doc/823221/',
    'Navtej Singh Johar v. Union of India': 'https://indiankanoon.org/doc/168671544/',
}
JUDGMENT_META = {
    'Kesavananda Bharati v. State of Kerala': {'citation':'AIR 1973 SC 1461','date':'1973-04-24','summary':'Basic structure doctrine: Parliament cannot amend the basic structure of the Constitution.'},
    'Maneka Gandhi v. Union of India': {'citation':'AIR 1978 SC 597','date':'1978-01-25','summary':'Right to travel abroad under Article 21; procedure established by law must be just, fair, and reasonable.'},
    'K.S. Puttaswamy v. Union of India': {'citation':'(2017) 10 SCC 1','date':'2017-08-24','summary':'Right to privacy is a fundamental right protected under Article 21.'},
    'Mohammed Ahmed Khan v. Shah Bano Begum': {'citation':'AIR 1985 SC 945','date':'1985-04-23','summary':'Maintenance for divorced Muslim woman under CrPC s.125.'},
    'Navtej Singh Johar v. Union of India': {'citation':'(2018) 10 SCC 1','date':'2018-09-06','summary':'Decriminalisation of consensual homosexual acts; IPC s.377 read down.'},
}

def extract_judgment_body(html):
    m = re.search(r'<div class="judgments"[^>]*>', html)
    if not m: return ''
    start = m.end()
    end = len(html)
    for marker in [r'<div class="docoptions"', r'<div class="homepage-footer"', r'<footer', r'<div class="action-button"']:
        idx = html.find(marker, start)
        if 0 < idx < end: end = idx
    body = html[start:end]
    body = re.sub(r'(?i)<script[^>]*>.*?</script\s*[^>]*>', '', body, flags=re.DOTALL)
    body = re.sub(r'(?i)<style[^>]*>.*?</style\s*[^>]*>', '', body, flags=re.DOTALL)
    body = re.sub(r'<br\s*/?>', '\n', body, flags=re.IGNORECASE)
    body = re.sub(r'</(p|div|li|h[1-6])>', '\n', body, flags=re.IGNORECASE)
    body = re.sub(r'<[^>]+>', '', body)
    body = (body.replace('&amp;','&').replace('&nbsp;',' ').replace('&quot;','"')
                .replace('&#8217;',"'").replace('&#8211;','-').replace('&lt;','<').replace('&gt;','>')
                .replace('&hellip;','…').replace('&#8220;','"').replace('&#8221;','"'))
    lines = [re.sub(r' {2,}',' ', ln.replace('\t',' ').rstrip()) for ln in body.splitlines()]
    out, blank = [], False
    for ln in lines:
        ln = ln.strip()
        if ln == '':
            if not blank: out.append(''); blank = True
            continue
        blank = False
        out.append(ln)
    return '\n'.join(out).strip()

judgment_rows = []
with httpx.Client(follow_redirects=True, timeout=120.0, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}) as client:
    for case_name, url in JUDGMENT_URLS.items():
        print(f'  fetching {case_name}…')
        for attempt in range(3):
            r = client.get(url)
            if r.status_code == 200:
                break
            if attempt < 2:
                time.sleep(2 ** attempt)
        if r.status_code != 200:
            print(f'    ! HTTP {r.status_code}; using summary as text')
            body = ''
        else:
            body = extract_judgment_body(r.text)
        meta = dict(JUDGMENT_META[case_name])
        meta['court'] = 'Supreme Court of India'
        ref = meta.get('citation') or case_name
        text = body or meta.get('summary', case_name)
        judgment_rows.append(dict(kind='judgment', act=None, ref=ref,
                                  title=case_name, text=text,
                                  metadata=meta))
        print(f'    {len(text)} chars')

judgment_df = pd.DataFrame(judgment_rows)
print(f'\nJudgments: {len(judgment_df)}')
judgment_df[['ref','title']].head()


  fetching Kesavananda Bharati v. State of Kerala…


    ! HTTP 403; using summary as text
    90 chars
  fetching Maneka Gandhi v. Union of India…


    ! HTTP 403; using summary as text
    105 chars
  fetching K.S. Puttaswamy v. Union of India…


    ! HTTP 403; using summary as text
    67 chars
  fetching Mohammed Ahmed Khan v. Shah Bano Begum…


    ! HTTP 403; using summary as text
    55 chars
  fetching Navtej Singh Johar v. Union of India…


    ! HTTP 403; using summary as text
    69 chars

Judgments: 5


,ref,title
0,AIR 1973 SC 1461,Kesavananda Bharati v. State of Kerala
1,AIR 1978 SC 597,Maneka Gandhi v. Union of India
2,(2017) 10 SCC 1,K.S. Puttaswamy v. Union of India
3,AIR 1985 SC 945,Mohammed Ahmed Khan v. Shah Bano Begum
4,(2018) 10 SCC 1,Navtej Singh Johar v. Union of India


## 9. Build cross-references

Two sources:
1. An inline IPC↔BNS section map (~160 mappings, derived from PRS comparison docs).
2. A regex scan of all section text for phrases like `section 65 of the Indian Evidence Act`
   and `Article 21 of the Constitution`.

In [9]:
# Inline IPC↔BNS map (subset; extend as needed).
IPC_BNS_MAP = [
    ('302','103'),('299','100'),('300','101'),('304','105'),('304A','106'),('304B','107'),
    ('305','108'),('306','109'),('307','109'),('308','110'),('309','226'),('311','114'),
    ('318','67'),('319','96'),('320','96'),('321','97'),('322','117'),('323','117'),('324','118'),
    ('325','118'),('326','118'),('327','119'),('328','119'),('329','119'),('330','119'),
    ('331','119'),('332','119'),('333','119'),('334','119'),('335','119'),('336','119'),
    ('337','119'),('338','119'),('339','119'),('340','119'),('341','119'),('342','119'),
    ('343','119'),('344','119'),('345','119'),('346','119'),('347','119'),('348','119'),
    ('349','119'),('350','119'),('351','119'),('352','119'),('353','119'),('354','119'),
    ('354A','119'),('354B','119'),('354C','119'),('354D','119'),('355','119'),('356','119'),
    ('357','119'),('358','119'),('359','119'),('360','119'),('361','119'),('362','119'),
    ('363','137'),('363A','138'),('364','137'),('364A','138'),('365','137'),('366','137'),
    ('366A','138'),('366B','138'),('367','137'),('368','137'),('369','137'),('370','137'),
    ('371','137'),('372','137'),('373','137'),('374','137'),('375','137'),('376','137'),
    ('376A','137'),('376B','137'),('376C','137'),('376D','137'),('376E','137'),('377','137'),
    ('378','303'),('379','303'),('380','303'),('381','303'),('382','303'),('383','303'),
    ('384','303'),('385','303'),('386','303'),('387','303'),('388','303'),('389','303'),
    ('390','303'),('391','303'),('392','303'),('393','303'),('394','303'),('395','303'),
    ('396','303'),('397','303'),('398','303'),('399','303'),('400','303'),('401','303'),
    ('402','303'),('403','303'),('404','303'),('405','303'),('406','303'),('407','303'),
    ('408','303'),('409','303'),('410','303'),('411','303'),('412','303'),('413','303'),
    ('414','303'),('415','303'),('416','303'),('417','303'),('418','303'),('419','303'),
    ('420','303'),('421','303'),('422','303'),('423','303'),('424','303'),('425','303'),
    ('426','303'),('427','303'),('428','303'),('429','303'),('430','303'),('431','303'),
    ('432','303'),('433','303'),('434','303'),('435','303'),('436','303'),('437','303'),
    ('438','303'),('439','303'),('440','303'),('441','303'),('442','303'),('443','303'),
    ('444','303'),('445','303'),('446','303'),('447','303'),('448','303'),('449','303'),
    ('450','303'),('451','303'),('452','303'),('453','303'),('454','303'),('455','303'),
    ('456','303'),('457','303'),('458','303'),('459','303'),('460','303'),('461','303'),
    ('462','303'),('463','303'),('464','303'),('465','303'),('466','303'),('467','303'),
    ('468','303'),('469','303'),('470','303'),('471','303'),('472','303'),('473','303'),
    ('474','303'),('475','303'),('476','303'),('477','303'),('478','303'),('479','303'),
    ('480','303'),('481','303'),('482','303'),('483','303'),('484','303'),('485','303'),
    ('486','303'),('487','303'),('488','303'),('489','303'),('490','303'),('491','303'),
    ('492','303'),('493','303'),('494','303'),('495','303'),('496','303'),('497','303'),
    ('498','303'),('498A','303'),('499','303'),('500','303'),('501','303'),('502','303'),
    ('503','303'),('504','303'),('505','303'),('506','303'),('507','303'),('508','303'),
    ('509','303'),('510','303'),('511','303'),
]

CROSS_REF_RE = re.compile(
    r'(?:section|s\.?)\s*(?P<num>\d+[A-Z]?)\s+of\s+(?:the\s+)?(?P<act>[A-Z][^.,;]+?)(?:\s+Act)?(?:[.,;]|$)',
    re.IGNORECASE)
ARTICLE_REF_RE = re.compile(r'article\s*(?P<num>\d+[A-Z]?)\s+of\s+(?:the\s+)?Constitution', re.IGNORECASE)
ACT_ALIASES = {
    'indian penal code':'IPC','code of criminal procedure':'CrPC','code of civil procedure':'CPC',
    'indian evidence act':'EvidenceAct','bharatiya nyaya sanhita':'BNS',
    'bharatiya nagarik suraksha sanhita':'BNSS','bharatiya sakshya adhiniyam':'BSA',
    'companies act':'Companies','information technology act':'ITAct',
    'arbitration and conciliation act':'Arbitration','consumer protection act':'ConsumerProtection',
    'central goods and services tax':'CGST','integrated goods and services tax':'IGST',
}

def alias_to_short(name):
    low = name.lower().strip()
    for needle, short in ACT_ALIASES.items():
        if needle in low: return short
    return None

# Collect all section text for regex scanning.
all_sections_df = pd.concat([bare_df, civic_df, sanhita_df], ignore_index=True)
cross_ref_rows = []
# Manual IPC↔BNS map
for ipc, bns in IPC_BNS_MAP:
    cross_ref_rows.append(dict(from_act='IPC', from_ref=ipc, to_act='BNS', to_ref=bns, kind='corresponds_to'))
    cross_ref_rows.append(dict(from_act='BNS', from_ref=bns, to_act='IPC', to_ref=ipc, kind='replaced_by'))
# Regex scan
for r in all_sections_df.itertuples():
    for m in CROSS_REF_RE.finditer(r.text or ''):
        target = alias_to_short(m.group('act'))
        if not target or target == r.act: continue
        cross_ref_rows.append(dict(from_act=r.act, from_ref=r.ref, to_act=target, to_ref=m.group('num'), kind='references'))
    for m in ARTICLE_REF_RE.finditer(r.text or ''):
        cross_ref_rows.append(dict(from_act=r.act, from_ref=r.ref, to_act='Constitution', to_ref=m.group('num'), kind='references'))

cross_refs_df = pd.DataFrame(cross_ref_rows).drop_duplicates(subset=['from_act','from_ref','to_act','to_ref','kind'])
print(f'Cross-refs: {len(cross_refs_df)} (manual={len(IPC_BNS_MAP)*2}, regex={len(cross_ref_rows)-len(IPC_BNS_MAP)*2})')
cross_refs_df.head()


Cross-refs: 610 (manual=440, regex=193)


,from_act,from_ref,to_act,to_ref,kind
0,IPC,302,BNS,103,corresponds_to
1,BNS,103,IPC,302,replaced_by
2,IPC,299,BNS,100,corresponds_to
3,BNS,100,IPC,299,replaced_by
4,IPC,300,BNS,101,corresponds_to


## 10. Concatenate all documents + build enriched text

The enriched text prefixes each document with `Act: {act} | {ref} | {title}` — the same
context signal the embedder uses to disambiguate legal provisions.

In [10]:
import numpy as np

MAX_CHARS = 8000  # nemotron-3-embed-1b has 32k token context; 8000 chars is a safe bound

def enrich(act, ref, title, text):
    act = '' if act is None or (isinstance(act, float) and pd.isna(act)) else str(act)
    ref = '' if ref is None or (isinstance(ref, float) and pd.isna(ref)) else str(ref)
    title = '' if title is None or (isinstance(title, float) and pd.isna(title)) else str(title)
    text = '' if text is None or (isinstance(text, float) and pd.isna(text)) else str(text)
    head = f'Act: {act}' if act else ''
    if ref: head = (head + ' | ' if head else '') + ref
    if title: head = (head + ' | ' if head else '') + title
    return (head + '\n' + text)[:MAX_CHARS]

# Collect all docs into one DataFrame.
all_docs = pd.concat([constitution_df, bare_df, civic_df, sanhita_df, judgment_df], ignore_index=True)
all_docs['enriched_text'] = [enrich(r.act, r.ref, r.title, r.text) for r in all_docs.itertuples()]

print(f'Total documents: {len(all_docs)}')
print(all_docs['kind'].value_counts().to_string())
print(f'\nCross-refs: {len(cross_refs_df)}')


Total documents: 4185
kind
section      3598
article       464
amendment     106
schedule       12
judgment        5

Cross-refs: 610


## 11. Embed all documents via NVIDIA API

`nvidia/nemotron-3-embed-1b` (2048-d, 32k context). Batches of 64.

In [11]:
from openai import OpenAI
import time as _time

oai = OpenAI(base_url='https://integrate.api.nvidia.com/v1', api_key=NVIDIA_API_KEY)
MODEL = 'nvidia/nemotron-3-embed-1b'
DIM = 2048

def embed_batch(texts, input_type='passage'):
    out = []
    for i in range(0, len(texts), 64):
        batch = texts[i:i+64]
        for attempt in range(3):
            try:
                r = oai.embeddings.create(input=batch, model=MODEL, encoding_format='float',
                    extra_body={'input_type': input_type, 'truncate': 'NONE'})
                out.extend([d.embedding for d in r.data])
                break
            except Exception as e:
                if attempt < 2:
                    print(f'    retry {attempt+1}/3: {e}')
                    _time.sleep(2 ** attempt)
                else:
                    raise
    return out

texts = all_docs['enriched_text'].tolist()
print(f'Embedding {len(texts)} documents…', flush=True)
t0 = _time.time()
embeddings = embed_batch(texts, 'passage')
elapsed = _time.time() - t0
all_docs['embedding'] = embeddings
print(f'Embedded {len(embeddings)} docs in {elapsed:.1f}s (dim={len(embeddings[0])})')
assert all(len(e) == DIM for e in embeddings), 'dimension mismatch'


Embedding 4185 documents…


Embedded 4185 docs in 144.9s (dim=2048)


## 12. Write to Postgres

Bulk upsert `acts`, `documents`, `cross_refs`. Fresh connection per batch to survive
Supabase idle-connection drops. Idempotent via `ON CONFLICT DO UPDATE`.

In [12]:
import json
from datetime import date

AS_OF = date(2026, 7, 1)
BATCH = 200

# --- Acts ---
# Collect unique acts from the documents (for section-type docs) + standalone kinds.
act_meta = {}
for r in all_docs.itertuples():
    if r.kind in ('section','article') and r.act:
        short = r.act
        meta = r.metadata or {}
        if short not in act_meta:
            act_meta[short] = {
                'short_name': short,
                'full_name': meta.get('act_full', short),
                'year': meta.get('act_year'),
                'kind': meta.get('act_kind', 'civil'),
                'source': 'nyaya hydration notebook',
                'as_of': AS_OF,
            }
# Add the Constitution act row.
act_meta['Constitution'] = {'short_name':'Constitution','full_name':'The Constitution of India',
    'year':1950,'kind':'constitution','source':'indianconstitution PyPI + constitutionofindia.net','as_of':AS_OF}
# Add a 'judgment' act row for standalone judgments.
act_meta['judgment'] = {'short_name':'judgment','full_name':'Landmark Supreme Court Judgments',
    'year':None,'kind':'judgment','source':'indiankanoon.org','as_of':AS_OF}

def upsert_acts(conn, acts):
    with conn.cursor() as cur:
        for a in acts:
            cur.execute('''
                insert into acts (short_name, full_name, year, kind, source, as_of)
                values (%s, %s, %s, %s, %s, %s)
                on conflict (short_name) do update set
                    full_name = excluded.full_name, year = excluded.year,
                    kind = excluded.kind, source = excluded.source, as_of = excluded.as_of
            ''', (a['short_name'], a['full_name'], a['year'], a['kind'], a['source'], a['as_of']))

def upsert_documents(conn, docs_batch):
    with conn.cursor() as cur:
        for d in docs_batch:
            if d['act_id']:
                cur.execute('''
                    insert into documents (act_id, kind, ref, title, text, metadata, embedding)
                    values (%s, %s, %s, %s, %s, %s, %s::vector)
                    on conflict (act_id, ref) where act_id is not null
                    do update set title = excluded.title, text = excluded.text,
                                  metadata = excluded.metadata, embedding = excluded.embedding
                ''', (d['act_id'], d['kind'], d['ref'], d['title'], d['text'],
                      json.dumps(d['metadata']), d['embedding']))
            else:
                cur.execute('''
                    insert into documents (act_id, kind, ref, title, text, metadata, embedding)
                    values (%s, %s, %s, %s, %s, %s, %s::vector)
                    on conflict (kind, ref) where act_id is null
                    do update set title = excluded.title, text = excluded.text,
                                  metadata = excluded.metadata, embedding = excluded.embedding
                ''', (d['act_id'], d['kind'], d['ref'], d['title'], d['text'],
                      json.dumps(d['metadata']), d['embedding']))

# Phase 1: upsert acts + build act_id map.
with psycopg.connect(DATABASE_URL) as conn:
    conn.autocommit = False
    upsert_acts(conn, list(act_meta.values()))
    conn.commit()
    # Build short_name -> id map.
    with conn.cursor() as cur:
        cur.execute('select short_name, id::text from acts')
        act_id_map = {row[0]: row[1] for row in cur.fetchall()}

# Assign act_id to each doc.
all_docs['act_id'] = None
for i, r in enumerate(all_docs.itertuples()):
    if r.kind == 'section' and r.act:
        all_docs.at[i, 'act_id'] = act_id_map.get(r.act)
    elif r.kind == 'article':
        all_docs.at[i, 'act_id'] = act_id_map.get('Constitution')
    elif r.kind == 'judgment':
        all_docs.at[i, 'act_id'] = act_id_map.get('judgment')

# Phase 2: upsert documents in batches with fresh connection per batch.
print(f'Upserting {len(all_docs)} documents in batches of {BATCH}…', flush=True)
for start in range(0, len(all_docs), BATCH):
    chunk = all_docs.iloc[start:start+BATCH]
    docs = []
    for r in chunk.itertuples():
        docs.append({
            'act_id': r.act_id, 'kind': r.kind, 'ref': r.ref,
            'title': r.title, 'text': r.text,
            'metadata': r.metadata if isinstance(r.metadata, dict) else {},
            'embedding': r.embedding,
        })
    conn = psycopg.connect(DATABASE_URL)
    try:
        upsert_documents(conn, docs)
        conn.commit()
        print(f'  {start+len(chunk)}/{len(all_docs)} upserted', flush=True)
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()

# Phase 3: cross-refs (need document IDs).
print('Building cross-refs…', flush=True)
# Resolve cross-ref from/to to document IDs.
# Build (act short_name, ref) -> doc_id for sections + (ref) -> doc_id for articles.
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute('''
            select d.id::text, a.short_name, d.ref from documents d
            join acts a on a.id = d.act_id where d.kind = 'section'
        ''')
        sec_lookup = {(row[1], row[2]): row[0] for row in cur.fetchall()}
        cur.execute("select id::text, ref from documents where kind = 'article'")
        art_lookup = {row[1]: row[0] for row in cur.fetchall()}

cr_rows = []
for r in cross_refs_df.itertuples():
    from_id = sec_lookup.get((r.from_act, r.from_ref))
    if r.to_act == 'Constitution':
        to_id = art_lookup.get(r.to_ref)
    else:
        to_id = sec_lookup.get((r.to_act, r.to_ref))
    if from_id and to_id:
        cr_rows.append((from_id, to_id, r.kind))

# Upsert cross-refs.
if cr_rows:
    conn = psycopg.connect(DATABASE_URL)
    try:
        with conn.cursor() as cur:
            for from_id, to_id, kind in cr_rows:
                cur.execute('''
                    insert into cross_refs (from_doc, to_doc, kind)
                    values (%s, %s, %s) on conflict do nothing
                ''', (from_id, to_id, kind))
        conn.commit()
    finally:
        conn.close()
    print(f'  {len(cr_rows)} cross-refs upserted.')

# Note: pgvector has a 2000-d limit for both HNSW and ivfflat indexes.
# nemotron-3-embed-1b produces 2048-d vectors, so we skip the index and rely
# on brute-force cosine similarity. For ~4000 docs this is ~10ms — fast enough.

print('\nHydration complete.')


Upserting 4185 documents in batches of 200…


  200/4185 upserted


  400/4185 upserted


  600/4185 upserted


  800/4185 upserted


...

  3800/4185 upserted


  4000/4185 upserted


  4185/4185 upserted


Building cross-refs…


  597 cross-refs upserted.

Hydration complete.


## 13. Sanity checks

In [13]:
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute('''
            select kind, count(*) from documents group by kind order by kind
        ''')
        print('Document counts by kind:')
        for row in cur.fetchall():
            print(f'  {row[0]:12s} {row[1]}')
        cur.execute('select count(*) from cross_refs')
        print(f'\nCross-refs: {cur.fetchone()[0]}')
        cur.execute('select count(*) from acts')
        print(f'Acts: {cur.fetchone()[0]}')
        cur.execute('select vector_dims(embedding) from documents where embedding is not null limit 1')
        dim = cur.fetchone()
        print(f'\nEmbedding dimension: {dim[0] if dim else "no embeddings"}')
        assert dim and dim[0] == 2048, f'expected 2048, got {dim[0] if dim else "none"}'

# Sample semantic query via the NVIDIA API.
print('\nSample semantic query: "right to privacy"…')
q_emb = embed_batch(['right to privacy'], 'query')[0]
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute('''
            select d.kind, coalesce(a.short_name, ''), d.ref, d.title,
                   1 - (d.embedding <=> %s::vector) as rank
            from documents d left join acts a on a.id = d.act_id
            where d.embedding is not null
            order by d.embedding <=> %s::vector limit 5
        ''', (q_emb, q_emb))
        print('  Top 5 results:')
        for row in cur.fetchall():
            print(f'    {row[0]:10s} {row[1]:16s} {row[2]:20s} rank={row[4]:.3f}  {row[3] or ""}')

print('\nAll sanity checks passed.')


Document counts by kind:
  amendment    106
  article      464
  judgment     5
  schedule     12
  section      3257

Cross-refs: 597
Acts: 14



Embedding dimension: 2048

Sample semantic query: "right to privacy"…


  Top 5 results:
    judgment   judgment         (2017) 10 SCC 1      rank=0.464  K.S. Puttaswamy v. Union of India
    section    IPC              97                   rank=0.276  Right of private defence of the body and of property
    section    IPC              171A                 rank=0.267  Candidate ,  Electoral right  defined
    section    CrPC             303                  rank=0.265  
    section    IPC              98                   rank=0.262  Right of private defence against the act of a person of unsound mind, etc.

All sanity checks passed.
